# 14. Modelo de lenguaje GRANDE ajustado (XLM-RoBERTa large) — Kaggle Notebook (GPU)

Idea: un LLM "preguntado" (sin entrenar) conoce el mundo pero no las convenciones del dataset (acierta ~70% en los casos dudosos, contra ~82% del modelo).
BETO/DistilBERT conocen el dataset pero son modelos chicos. Acá se **ajusta un modelo ~5 veces más grande** (560M parámetros, preentrenado en 100 idiomas)
para tener las dos cosas: conocimiento previo + convenciones del dataset.

- Mismo texto que la notebook 10 + cantidad de publicaciones del vendedor.
- Mismos 5 folds (StratifiedKFold, seed 42) → `oof_xlmr.npy` comparable con BERT/BETO.
- Guarda después de cada fold. ⏱ ~35-45 min por fold en T4.
- Control: si al final de la época 1 la accuracy de validación está cerca de 0.54, el modelo colapsó → bajar `LR` a 7e-6 y volver a correr ese fold.



In [ ]:
import os, glob, time, re, json
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold

MODELO = 'xlm-roberta-large'
MAX_LEN, BATCH, EPOCHS, LR, SEED = 128, 16, 3, 1e-5, 42
FOLDS_A_CORRER = [0, 1, 2, 3, 4]      # si la sesión se corta, volver a correr sólo los que faltan
OUT = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print('device:', device, '|', torch.cuda.get_device_name(0) if device == 'cuda' else '')

In [ ]:
def buscar(nombre):
    c = glob.glob(f'/kaggle/input/**/{nombre}', recursive=True) + glob.glob(nombre)
    assert c, f'No encuentro {nombre}: agregá la competencia como input'
    return c[0]
tr = pd.read_json(buscar('train_data.jsonlines'), lines=True)
te = pd.read_json(buscar('test_data.jsonlines'), lines=True)
y = (tr.condition == 'used').astype(int).values
print(tr.shape, te.shape)
_vc = pd.concat([tr.seller_id, te.seller_id]).value_counts()   # publicaciones del vendedor en train+test (no usa el target)
tr['nv_vend'] = tr.seller_id.map(_vc).values; te['nv_vend'] = te.seller_id.map(_vc).values


In [ ]:
def a_texto(df):
    out = []
    for r in df.itertuples(index=False):
        ship = r.shipping if isinstance(r.shipping, dict) else {}
        attrs = ', '.join(f"{a.get('name','')}: {a.get('value_name','')}" for a in (r.attributes or [])[:12])
        var = ', '.join(sorted({f"{c.get('name')}: {c.get('value_name')}" for v in (r.variations or []) for c in v.get('attribute_combinations', [])})[:8])
        pagos = ', '.join(m['description'] for m in (r.non_mercado_pago_payment_methods or []))
        g = r.warranty[:80] if isinstance(r.warranty, str) else 'sin dato'
        out.append(f"{r.title} | garantía: {g} | tipo: {r.listing_type_id} | {r.buying_mode} | precio: {int(round(r.price))} {r.currency_id}"
                   f" | stock: {r.initial_quantity} | vendidos: {r.sold_quantity} | fotos: {len(r.pictures or [])}"
                   f" | variaciones: {len(r.variations or [])} {var} | atributos: {attrs}"
                   f" | envío: {ship.get('mode','')}{' gratis' if ship.get('free_shipping') else ''} | relist: {r.automatic_relist} | pagos: {pagos} | publicaciones del vendedor: {r.nv_vend}")
    return out
T, Tt = a_texto(tr), a_texto(te)
print(T[0][:300]); print(T[2][:300])

In [ ]:
tok = AutoTokenizer.from_pretrained(MODELO)
def loader(textos, labels=None, shuffle=False, bs=BATCH):
    idx = list(range(len(textos)))
    def collate(b):
        enc = tok([textos[i] for i in b], truncation=True, max_length=MAX_LEN, padding=True, return_tensors='pt')
        if labels is not None: enc['labels'] = torch.tensor([labels[i] for i in b])
        return enc
    return DataLoader(idx, batch_size=bs, shuffle=shuffle, collate_fn=collate)
usar_amp = device == 'cuda'
def predecir(model, textos):
    model.eval(); out = []
    with torch.no_grad():
        for b in loader(textos, bs=256):
            b = {k: v.to(device) for k, v in b.items()}
            with torch.autocast('cuda', dtype=torch.float16, enabled=usar_amp):
                logits = model(**b).logits
            out.append(torch.softmax(logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out)

In [ ]:
folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(T, y))
for f in FOLDS_A_CORRER:
    a, b = folds[f]; t0 = time.time()
    Ta, ya = [T[i] for i in a], y[a].tolist()
    model = AutoModelForSequenceClassification.from_pretrained(MODELO, num_labels=2).to(device)
    dl = loader(Ta, ya, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.10 * len(dl) * EPOCHS), len(dl) * EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=usar_amp)
    for ep in range(EPOCHS):
        model.train()
        for i, batch in enumerate(dl):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast('cuda', dtype=torch.float16, enabled=usar_amp):
                loss = model(**batch).loss
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad()
            if i % 500 == 0: print(f'fold {f} ep {ep} paso {i}/{len(dl)} loss {loss.item():.3f} ({(time.time()-t0)/60:.1f} min)', flush=True)
        pb = predecir(model, [T[i] for i in b])
        print(f'   fold {f} fin época {ep}: accuracy val {((pb >= .5) == y[b]).mean():.4f}', flush=True)
    np.save(f'{OUT}/oof_xlmr_fold{f}.npy', pb); np.save(f'{OUT}/test_xlmr_fold{f}.npy', predecir(model, Tt))
    print(f'>>> fold {f}: accuracy {((pb >= .5) == y[b]).mean():.4f}  ({(time.time()-t0)/60:.1f} min)', flush=True)
    del model, opt; torch.cuda.empty_cache() if device == 'cuda' else None

### Juntar los 5 folds (se puede correr aunque los folds se hayan hecho en sesiones distintas)

In [ ]:
oof = np.full(len(y), np.nan); tests = []
for f, (a, b) in enumerate(folds):
    p = f'{OUT}/oof_xlmr_fold{f}.npy'
    if os.path.exists(p): oof[b] = np.load(p); tests.append(np.load(f'{OUT}/test_xlmr_fold{f}.npy'))
ok = ~np.isnan(oof)
print(f'folds listos: {len(tests)}/5 | XLM-R large accuracy OOF: {((oof[ok] >= .5) == y[ok]).mean():.4f}  (DistilBERT: 0.9041 | BETO: 0.9075)')
if ok.all():
    np.save(f'{OUT}/oof_xlmr.npy', oof); np.save(f'{OUT}/test_xlmr.npy', np.mean(tests, 0)); print('guardados oof_xlmr.npy y test_xlmr.npy')